<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 자연어 처리와 LLM · 06. T5

## Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer

- **원 논문:** [Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer](https://arxiv.org/abs/1910.10683)
- **저자 / 발표:** Colin Raffel et al. · JMLR 2020 (2019)
- **난이도 / 예상 시간:** 중급 · 약 120분
- **선수 지식:** Python, PyTorch, 확률·선형대수·sequence model의 기초
- **로컬 학습 데이터:** `data/field_curriculum/nlp_corpus.json`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** text-to-text 형식, span sentinel target, relative position bias를 재현한다.

**축소하거나 생략한 부분:** C4와 최대 11B parameter 대신 60개 3-token sequence와 작은 Transformer를 쓰며, 48개 train/12개 held-out 행을 분리한다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §3.1.4 and Fig. 2: span replacement with sentinel tokens | Task 1 | sentinel input/target sequence |
| §2.1: encoder-decoder attention and relative position embeddings | Task 2 explicit encoder/causal decoder/cross-attention | bias shape, nonzero gradient, and future invariance |
| §3.3.4: span-corruption objective | Task 3 | denoising accuracy |

## 핵심 아이디어와 수식

$$L_{span}=-\sum_t\log p(y_t\mid y_{<t},\widetilde{x})$$

**기호 해설:** x-tilde는 sentinel로 span을 바꾼 입력, y는 sentinel과 제거 span을 잇는 target이다.

**코드 대응:** Task 1이 span pair, Task 2가 logits에 relative bias를 더하는 encoder/causal decoder/cross-attention, Task 3이 denoising CE를 맡는다.

**입력과 출력 shape:** train source [48,2], decoder [48,4]와 held-out source [12,2]를 분리해 logits [B,4,V]와 token accuracy를 계산한다.

**포트폴리오 구현 범위:** 여러 span과 bucketed bias 대신 한 span/clip distance를 쓰되 text-to-text 목적은 유지한다.

각 단계는 데이터 전처리와 shape 확인부터 논문 고유 class, `forward`, 목적함수,
`train_step`, 평가까지 이어진다. 핵심 수식을 한 번의 고수준 호출로 감추지 않고
중간 tensor를 확인할 수 있게 분리한다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.manual_seed,json.loads,Path.read_text,torch.tensor,ACCELERATOR.move -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 5개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.manual_seed(seed)</code></summary>

#### `torch.manual_seed(seed)`

- **역할:** PyTorch 난수 생성기의 시드를 설정합니다.
- **입력·반환:** 정수 시드를 받고 `Generator`를 반환합니다.
- **이 셀에서 쓰는 이유:** 가중치 초기화와 텐서 샘플링을 반복 가능하게 만듭니다.
- **주의점:** 완전한 결정성에는 알고리즘·장치 설정도 추가로 필요할 수 있습니다.
- **공식 문서:** [torch.manual_seed](https://docs.pytorch.org/docs/stable/generated/torch.manual_seed.html)

</details>

<details>
<summary><code>json.loads(s, ...)</code></summary>

#### `json.loads(s, ...)`

- **역할:** JSON 문자열을 Python 객체로 역직렬화합니다.
- **입력·반환:** 문자열·bytes를 받고 dict·list 등의 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 메모리에 있는 JSON line이나 응답을 구조화합니다.
- **주의점:** 입력이 한 문서인지 JSON Lines의 한 줄인지 구분해야 합니다.
- **공식 문서:** [json.loads](https://docs.python.org/3/library/json.html#json.loads)

</details>

<details>
<summary><code>path.read_text(encoding=None, errors=None)</code></summary>

#### `path.read_text(encoding=None, errors=None)`

- **역할:** 텍스트 파일 전체를 문자열로 읽습니다.
- **입력·반환:** 인코딩 설정을 받고 문자열을 반환합니다.
- **이 셀에서 쓰는 이유:** 작은 설정·말뭉치·메타데이터 파일을 명시적 인코딩으로 읽습니다.
- **주의점:** 학습용 대용량 파일에는 전체 메모리 적재가 부담될 수 있습니다.
- **공식 문서:** [Path.read_text](https://docs.python.org/3/library/pathlib.html#pathlib.Path.read_text)

</details>

<details>
<summary><code>torch.tensor(data, dtype=None, device=None)</code></summary>

#### `torch.tensor(data, dtype=None, device=None)`

- **역할:** 데이터를 복사해 새 PyTorch 텐서를 만듭니다.
- **입력·반환:** 배열형 데이터와 선택적 dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 상수·label·축소 실험 입력을 명시적으로 구성합니다.
- **주의점:** 기존 텐서에서 호출하면 복사가 생기므로 경우에 따라 `as_tensor`가 낫습니다.
- **공식 문서:** [torch.tensor](https://docs.pytorch.org/docs/stable/generated/torch.tensor.html)

</details>

<details>
<summary><code>ACCELERATOR.move(value)</code></summary>

#### `ACCELERATOR.move(value)`

- **역할:** 텐서나 모듈을 선택된 실행 장치로 옮기는 프로젝트 도우미입니다.
- **입력·반환:** 이동할 값을 받고 같은 논리 값을 장치에 배치해 반환합니다.
- **이 셀에서 쓰는 이유:** 모델과 batch가 서로 다른 장치에 놓이는 오류를 막습니다.
- **주의점:** 컨테이너 재귀 처리 여부와 dtype 변경 여부는 프로젝트 구현을 확인해야 합니다.
- **공식 문서:** 프로젝트 내부 도우미이므로 정의의 docstring을 확인합니다.

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
import json
import math
import re
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(20260814)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device


def locate(relative_path):
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(relative_path)


dataset_path = locate("data/field_curriculum/nlp_corpus.json")
corpus = json.loads(dataset_path.read_text(encoding="utf-8"))
sentences = corpus["sentences"]
translation_pairs = corpus["translation_pairs"]
documents = corpus["documents"]
queries = corpus["queries"]
SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>", "<mask>", "<sentinel>"]
sentence_words = {word.lower() for sentence in sentences for word in sentence.split()}
translation_words = {
    word.lower()
    for pair in translation_pairs
    for side in ("source", "target")
    for word in pair[side].split()
}
words = sorted(sentence_words | translation_words)
itos = SPECIAL_TOKENS + words
stoi = {word: index for index, word in enumerate(itos)}
PAD, BOS, EOS, MASK, SENTINEL = range(5)
VOCAB_SIZE = len(itos)


def encode(text, length=4):
    token_ids = [stoi[word.lower()] for word in text.split()]
    token_ids = token_ids[:length]
    return token_ids + [PAD] * (length - len(token_ids))


sentence_ids = torch.tensor([encode(text) for text in sentences])
sentiment_labels = torch.tensor(corpus["sentiment_labels"], dtype=torch.long)
source_ids = torch.tensor([encode(pair["source"], 3) for pair in translation_pairs])
target_ids = torch.tensor([encode(pair["target"], 3) for pair in translation_pairs])
sentence_ids, sentiment_labels = ACCELERATOR.move(
    sentence_ids,
    sentiment_labels,
)
source_ids, target_ids = ACCELERATOR.move(source_ids, target_ids)
assert sentence_ids.shape == (360, 4)
assert source_ids.shape == target_ids.shape == (60, 3)
print(ACCELERATOR.summary())
print(dataset_path.name, f"sentences={len(sentences)}, vocab={VOCAB_SIZE}")

### 구현 단계 1 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §3.1.4 and Fig. 2: span replacement with sentinel tokens
- **노트북에서 구현할 부분:** Task 1
- **구현 이유:** 원문의 확률 분해나 tensor 변환을 작은 입력에서 직접
  관찰하고, 고수준 API 뒤에 핵심 알고리즘을 숨기지 않기 위해서다.
- **완료 증거:** sentinel input/target sequence

아래 TODO 전에 입력 token/feature shape와 수식의 축을 먼저 확인한다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=dataclass,torch.stack,torch.cat -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 3개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>@dataclass(frozen=False, slots=False, ...)</code></summary>

#### `@dataclass(frozen=False, slots=False, ...)`

- **역할:** type annotation을 바탕으로 초기화·표현 메서드 등을 생성합니다.
- **입력·반환:** class decorator 설정을 받고 변환된 class를 반환합니다.
- **이 셀에서 쓰는 이유:** 실험 설정과 결과 record의 필드를 명시적으로 정의합니다.
- **주의점:** mutable 기본값은 `default_factory`를 쓰고 equality·frozen 의미를 확인합니다.
- **공식 문서:** [dataclass](https://docs.python.org/3/library/dataclasses.html#dataclasses.dataclass)

</details>

<details>
<summary><code>torch.stack(tensors, dim=0)</code></summary>

#### `torch.stack(tensors, dim=0)`

- **역할:** 새 축을 만들어 텐서를 쌓습니다.
- **입력·반환:** 텐서와 축·연산 설정을 받고 변환된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 수식의 축 관계와 batch 연산을 코드로 직접 표현합니다.
- **주의점:** 입력 shape과 broadcasting 규칙을 셀의 shape assertion으로 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\operatorname{shape}(Y)=(d_1,\ldots,d_{a-1},R,d_a,\ldots,d_K)
$$

- **기호:** $R$은 쌓는 tensor 개수이고 $a$는 새 축이 들어갈 위치입니다.
- **수식 → 코드:** `stack`은 각 입력을 새 index 하나로 배치해 새로운 축을 만듭니다.
- **직관:** 연결할 기존 축을 고르는 `cat`과 달리 축의 개수가 하나 늘어납니다.
- **shape 확인:** 모든 입력 shape이 완전히 같아야 합니다.
- **공식 문서:** [torch.stack](https://docs.pytorch.org/docs/stable/generated/torch.stack.html)

</details>

<details>
<summary><code>torch.cat(tensors, dim=0)</code></summary>

#### `torch.cat(tensors, dim=0)`

- **역할:** 기존 축을 따라 텐서를 연결합니다.
- **입력·반환:** 텐서와 축·연산 설정을 받고 변환된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 수식의 축 관계와 batch 연산을 코드로 직접 표현합니다.
- **주의점:** 입력 shape과 broadcasting 규칙을 셀의 shape assertion으로 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
d_{\mathrm{out}}^{(a)}=\sum_{r=1}^{R}d_r^{(a)}
$$

- **기호:** $a$는 연결할 축이고 $R$은 입력 tensor 개수입니다.
- **수식 → 코드:** `dim=a` 축의 길이만 더하고 나머지 축은 그대로 유지합니다.
- **직관:** 이미 존재하는 축을 길게 이어 붙이는 연산입니다.
- **shape 확인:** 연결 축을 제외한 모든 shape과 dtype·device가 같아야 합니다.
- **공식 문서:** [torch.cat](https://docs.pytorch.org/docs/stable/generated/torch.cat.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
@dataclass
class T5Config:
    model_dim: int = 24
    heads: int = 4
    feedforward_dim: int = 48
    max_distance: int = 4
    learning_rate: float = 0.02
    steps: int = 32


def make_span_corruption(source, sentinel_id, eos_id, bos_id):
    sentinel = source.new_full((len(source),), sentinel_id)
    eos = source.new_full((len(source),), eos_id)
    corrupted = torch.stack((source[:, 0], sentinel), dim=1)
    target = torch.stack(
        (sentinel, source[:, 1], source[:, 2], eos),
        dim=1,
    )
    decoder_inputs = torch.cat(
        (source.new_full((len(source), 1), bos_id), target[:, :-1]),
        dim=1,
    )
    return corrupted, target, decoder_inputs


config = T5Config()
split_generator = torch.Generator().manual_seed(32)
translation_order = torch.randperm(
    len(source_ids),
    generator=split_generator,
).to(source_ids.device)
train_translation_index = translation_order[:48]
test_translation_index = translation_order[48:]
assert not bool(
    torch.isin(
        train_translation_index,
        test_translation_index,
    ).any()
)
train_span_input, train_span_target, train_decoder_inputs = make_span_corruption(
    source_ids[train_translation_index],
    SENTINEL,
    EOS,
    BOS,
)
test_span_input, test_span_target, test_decoder_inputs = make_span_corruption(
    source_ids[test_translation_index],
    SENTINEL,
    EOS,
    BOS,
)
assert train_span_input.shape == (48, 2)
assert test_span_input.shape == (12, 2)
assert train_span_target.shape == train_decoder_inputs.shape == (48, 4)
assert test_span_target.shape == test_decoder_inputs.shape == (12, 4)

### 구현 단계 2 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §2.1: encoder-decoder attention and relative position embeddings
- **노트북에서 구현할 부분:** Task 2 explicit encoder/causal decoder/cross-attention
- **구현 이유:** 원문의 확률 분해나 tensor 변환을 작은 입력에서 직접
  관찰하고, 고수준 API 뒤에 핵심 알고리즘을 숨기지 않기 위해서다.
- **완료 증거:** bias shape, nonzero gradient, and future invariance

아래 TODO 전에 입력 token/feature shape와 수식의 축을 먼저 확인한다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.Embedding,torch.arange,nn.Linear,Tensor.reshape,Tensor.transpose,torch.matmul,Tensor.masked_fill,Tensor.softmax,nn.LayerNorm,nn.Sequential,nn.ReLU,torch.ones,torch.bool,Tensor.backward,Optimizer.zero_grad,torch.no_grad,torch.allclose -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 17개 · 수식 11개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.Embedding(num_embeddings, embedding_dim, ...)</code></summary>

#### `nn.Embedding(num_embeddings, embedding_dim, ...)`

- **역할:** 정수 ID를 학습 가능한 벡터로 조회합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
e_i=E[i],\qquad E\in\mathbb{R}^{V\times D}
$$

- **기호:** $i$는 정수 ID, $V$는 vocabulary 크기, $D$는 embedding 차원입니다.
- **수식 → 코드:** forward는 one-hot 행렬곱을 만들지 않고 weight 행 $E[i]$를 직접 조회합니다.
- **직관:** 범주 ID를 학습 가능한 연속 벡터 좌표로 바꿉니다.
- **shape 확인:** 입력 `[...] -> 출력 [..., D]`; 입력 dtype은 보통 `torch.int64`입니다.
- **공식 문서:** [nn.Embedding](https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html)

</details>

<details>
<summary><code>torch.arange(start=0, end, step=1, ...)</code></summary>

#### `torch.arange(start=0, end, step=1, ...)`

- **역할:** 일정 간격의 1차원 텐서를 만듭니다.
- **입력·반환:** 범위와 간격을 받고 1차원 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** position·index·time step을 벡터화해 만듭니다.
- **주의점:** 부동소수점 step은 반올림 때문에 원소 수가 예상과 다를 수 있습니다.
- **공식 문서:** [torch.arange](https://docs.pytorch.org/docs/stable/generated/torch.arange.html)

</details>

<details>
<summary><code>nn.Linear(in_features, out_features, bias=True)</code></summary>

#### `nn.Linear(in_features, out_features, bias=True)`

- **역할:** 완전연결 affine 변환을 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y=xW^{\mathsf T}+b
$$

- **기호:** $x$는 입력, $W$는 weight, $b$는 bias, $y$는 출력입니다.
- **수식 → 코드:** `in_features`와 `out_features`가 각각 $W$의 두 축을 결정합니다.
- **직관:** 각 출력 feature는 모든 입력 feature의 학습 가능한 가중합입니다.
- **shape 확인:** `[..., D_in] -> [..., D_out]`, weight는 `[D_out, D_in]`입니다.
- **공식 문서:** [nn.Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html)

</details>

<details>
<summary><code>tensor.reshape(*shape)</code></summary>

#### `tensor.reshape(*shape)`

- **역할:** 원소 수를 유지하며 텐서 shape을 바꿉니다.
- **입력·반환:** 새 shape을 받고 가능하면 view, 아니면 복사된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** batch·head·sequence 축을 논문 수식의 shape에 맞춥니다.
- **주의점:** 원소 수가 같아야 하며 메모리 공유 여부에 의존하면 안 됩니다.

##### 관련 수식과 코드 연결

$$
\prod_{k=1}^{K} d_k=\prod_{j=1}^{J} d'_j
$$

- **기호:** $d_k$는 원래 각 축 크기, $d'_j$는 바꾼 뒤 각 축 크기입니다.
- **수식 → 코드:** `reshape`·`view`·`flatten`은 원소의 총개수를 보존하면서 축 해석만 바꿉니다.
- **직관:** 데이터 값을 계산하는 연산이 아니라 같은 원소를 다른 좌표계로 보는 연산입니다.
- **shape 확인:** 예: `[B, H, T, D] -> [B, T, H*D]`; 왼쪽과 오른쪽 원소 수가 같아야 합니다.
- **공식 문서:** [Tensor.reshape](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.reshape.html)

</details>

<details>
<summary><code>tensor.transpose(dim0, dim1)</code></summary>

#### `tensor.transpose(dim0, dim1)`

- **역할:** 두 축의 위치를 맞바꿉니다.
- **입력·반환:** 두 축 번호를 받고 축이 교환된 view를 반환합니다.
- **이 셀에서 쓰는 이유:** attention 행렬곱 등에서 축 순서를 맞춥니다.
- **주의점:** 반환 텐서는 비연속일 수 있어 이후 `view` 전에 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
Y_{i_{\pi(1)},\ldots,i_{\pi(K)}}=X_{i_1,\ldots,i_K}
$$

- **기호:** $\pi$는 축의 새 순서를 나타내는 permutation입니다.
- **수식 → 코드:** `transpose`는 두 축만 바꾸고 `permute`는 모든 축의 새 순서를 명시합니다.
- **직관:** 원소 값을 바꾸지 않고 각 index가 의미하는 축의 위치만 다시 배치합니다.
- **shape 확인:** 예: `[B,T,H,D] -> [B,H,T,D]`; 결과는 비연속 view일 수 있습니다.
- **공식 문서:** [Tensor.transpose](https://docs.pytorch.org/docs/stable/generated/torch.transpose.html)

</details>

<details>
<summary><code>torch.matmul(input, other)</code></summary>

#### `torch.matmul(input, other)`

- **역할:** 벡터·행렬·batch 행렬곱을 수행합니다.
- **입력·반환:** 텐서와 축·연산 설정을 받고 변환된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 수식의 축 관계와 batch 연산을 코드로 직접 표현합니다.
- **주의점:** 입력 shape과 broadcasting 규칙을 셀의 shape assertion으로 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
C_{ij}=\sum_{k=1}^{K}A_{ik}B_{kj}
$$

- **기호:** $k$는 곱한 뒤 사라지는 축이고 $i,j$는 결과에 남는 축입니다.
- **수식 → 코드:** `matmul`은 마지막 두 축에 이 합을 적용하고, `bmm`은 batch별로, `einsum`은 문자열로 축 관계를 지정합니다.
- **직관:** 공통 축 $k$의 정보를 가중합해 두 표현 사이의 상호작용을 만듭니다.
- **shape 확인:** `[..., I, K] @ [..., K, J] -> [..., I, J]`이며 앞쪽 batch 축은 broadcast될 수 있습니다.
- **공식 문서:** [torch.matmul](https://docs.pytorch.org/docs/stable/generated/torch.matmul.html)

</details>

<details>
<summary><code>tensor.masked_fill(mask, value)</code></summary>

#### `tensor.masked_fill(mask, value)`

- **역할:** mask가 참인 위치를 지정 값으로 바꿉니다.
- **입력·반환:** 불리언 mask와 값을 받고 새 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** padding이나 미래 token의 attention score를 선택에서 제외합니다.
- **주의점:** mask는 입력에 broadcast 가능해야 하고 softmax 전 값은 충분히 작아야 합니다.

##### 관련 수식과 코드 연결

$$
s'_{ij}=\begin{cases}s_{ij},&M_{ij}=0\\-\infty,&M_{ij}=1\end{cases}\quad\Rightarrow\quad\operatorname{softmax}(s')_{ij}=0\;\text{if }M_{ij}=1
$$

- **기호:** $s$는 score, $M$은 제외할 위치가 참인 mask입니다.
- **수식 → 코드:** attention에서는 mask 위치를 매우 작은 값으로 채운 뒤 softmax를 적용합니다.
- **직관:** 확률을 나중에 지우는 대신 정규화 분모에도 들어가지 못하게 합니다.
- **shape 확인:** mask는 score에 broadcast 가능해야 하며 보통 `[B, H, T_q, T_k]`를 사용합니다.
- **공식 문서:** [Tensor.masked_fill](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.masked_fill.html)

</details>

<details>
<summary><code>tensor.softmax(dim, dtype=None)</code></summary>

#### `tensor.softmax(dim, dtype=None)`

- **역할:** 지정 축의 값을 합이 1인 확률형 가중치로 변환합니다.
- **입력·반환:** 텐서와 축을 받고 같은 shape의 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** method 문법으로 attention score나 class logit을 정규화합니다.
- **주의점:** `dim`이 확률 합을 낼 축이며 입력은 일반적으로 raw logit이어야 합니다.

##### 관련 수식과 코드 연결

$$
p_i=\frac{e^{z_i}}{\sum_j e^{z_j}},\qquad \log p_i=z_i-\log\sum_j e^{z_j}
$$

- **기호:** $z_i$는 logit, $p_i$는 선택한 축에서 합이 1인 확률입니다.
- **수식 → 코드:** 코드의 `dim`이 분모의 합을 계산할 $j$축을 결정합니다.
- **직관:** 각 logit을 다른 후보와 비교 가능한 상대 확률로 바꿉니다.
- **shape 확인:** 입력과 출력 shape은 같고 선택한 축의 softmax 값 합은 1입니다.
- **공식 문서:** [Tensor.softmax](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.softmax.html)

</details>

<details>
<summary><code>nn.LayerNorm(normalized_shape, ...)</code></summary>

#### `nn.LayerNorm(normalized_shape, ...)`

- **역할:** 마지막 feature 축들을 표준화합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y_i=\gamma_i\frac{x_i-\mu}{\sqrt{\sigma^2+\epsilon}}+\beta_i
$$

- **기호:** $\mu,\sigma^2$는 한 sample의 지정 feature 축 평균·분산입니다.
- **수식 → 코드:** `normalized_shape`의 마지막 축들에서 통계를 내고 학습 가능한 scale과 bias를 적용합니다.
- **직관:** batch 크기와 무관하게 각 token·sample의 feature scale을 안정화합니다.
- **shape 확인:** 입력과 출력 shape은 같고 $\gamma,\beta$는 `normalized_shape`를 따릅니다.
- **공식 문서:** [nn.LayerNorm](https://docs.pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html)

</details>

<details>
<summary><code>nn.Sequential(*modules)</code></summary>

#### `nn.Sequential(*modules)`

- **역할:** 모듈을 순서대로 연결합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.
- **공식 문서:** [nn.Sequential](https://docs.pytorch.org/docs/stable/generated/torch.nn.Sequential.html)

</details>

<details>
<summary><code>nn.ReLU(inplace=False)</code></summary>

#### `nn.ReLU(inplace=False)`

- **역할:** 음수 값을 0으로 만드는 활성함수를 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\operatorname{ReLU}(x)=\max(0,x)
$$

- **기호:** $x$는 선형 변환 뒤 activation입니다.
- **수식 → 코드:** 코드가 음수 원소를 0으로 치환하고 양수는 그대로 통과시킵니다.
- **직관:** 단순하지만 여러 선형 층을 비선형 함수로 조합할 수 있게 합니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [nn.ReLU](https://docs.pytorch.org/docs/stable/generated/torch.nn.ReLU.html)

</details>

<details>
<summary><code>torch.ones(*size, dtype=None, device=None)</code></summary>

#### `torch.ones(*size, dtype=None, device=None)`

- **역할:** 모든 원소가 1인 텐서를 만듭니다.
- **입력·반환:** shape·dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 초기 weight·mask·스케일을 구성합니다.
- **주의점:** 주변 연산과 dtype·device가 같은지 확인해야 합니다.
- **공식 문서:** [torch.ones](https://docs.pytorch.org/docs/stable/generated/torch.ones.html)

</details>

<details>
<summary><code>torch.bool</code></summary>

#### `torch.bool`

- **역할:** 불리언 dtype
- **입력·반환:** 값이 아니라 텐서 원소의 저장 형식으로 사용됩니다.
- **이 셀에서 쓰는 이유:** mask와 조건 결과를 표현합니다.
- **주의점:** 연산마다 허용 dtype이 다르고 변환 시 정밀도나 메모리가 달라집니다.
- **공식 문서:** [torch.bool](https://docs.pytorch.org/docs/stable/tensor_attributes.html#torch.dtype)

</details>

<details>
<summary><code>loss.backward()</code></summary>

#### `loss.backward()`

- **역할:** 스칼라 loss에서 계산 그래프를 거슬러 gradient를 계산합니다.
- **입력·반환:** 스칼라 텐서에서는 보통 인자 없이 호출하며 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** optimizer update에 필요한 각 파라미터의 `.grad`를 채웁니다.
- **주의점:** 같은 그래프를 다시 역전파하려면 별도 보존 설정이 필요합니다.

##### 관련 수식과 코드 연결

$$
\frac{\partial L}{\partial x}=\frac{\partial L}{\partial y}\frac{\partial y}{\partial x},\qquad x.\mathrm{grad}\leftarrow x.\mathrm{grad}+\nabla_xL
$$

- **기호:** $L$은 scalar loss이고 $x$는 계산 그래프의 leaf parameter입니다.
- **수식 → 코드:** `backward()`는 연쇄법칙으로 gradient를 계산해 각 parameter의 `.grad`에 누적합니다.
- **직관:** 파라미터를 직접 바꾸는 호출이 아니라 update에 필요한 미분값을 준비하는 호출입니다.
- **shape 확인:** 각 `.grad`는 대응 parameter와 같은 shape이며 scalar가 아닌 출력에는 seed gradient가 필요합니다.
- **공식 문서:** [Tensor.backward](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.backward.html)

</details>

<details>
<summary><code>optimizer.zero_grad(set_to_none=True)</code></summary>

#### `optimizer.zero_grad(set_to_none=True)`

- **역할:** 파라미터에 누적된 gradient를 비웁니다.
- **입력·반환:** 초기화 방식을 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 현재 mini-batch의 gradient만으로 update하도록 학습 단계를 시작합니다.
- **주의점:** 호출하지 않으면 PyTorch gradient가 기본적으로 누적됩니다.
- **공식 문서:** [Optimizer.zero_grad](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.zero_grad.html)

</details>

<details>
<summary><code>torch.no_grad()</code></summary>

#### `torch.no_grad()`

- **역할:** 블록 안에서 autograd 기록을 끄는 context manager입니다.
- **입력·반환:** 보통 `with`에 사용하며 블록의 텐서는 기본적으로 gradient를 기록하지 않습니다.
- **이 셀에서 쓰는 이유:** 평가나 고정 teacher 추론의 메모리와 연산 부담을 줄입니다.
- **주의점:** `model.eval()`과 역할이 다르므로 평가 모드가 필요하면 둘 다 사용합니다.
- **공식 문서:** [torch.no_grad](https://docs.pytorch.org/docs/stable/generated/torch.no_grad.html)

</details>

<details>
<summary><code>torch.allclose(input, other, rtol=1e-5, atol=1e-8, equal_nan=False)</code></summary>

#### `torch.allclose(input, other, rtol=1e-5, atol=1e-8, equal_nan=False)`

- **역할:** 두 텐서의 모든 원소가 허용 오차 안에서 가까운지 검사합니다.
- **입력·반환:** 두 텐서와 오차 설정을 받고 Python `bool` 하나를 반환합니다.
- **이 셀에서 쓰는 이유:** 장치나 구현을 바꾼 뒤 수치 결과가 사실상 같은지 확인합니다.
- **주의점:** 정확한 동일성 검사가 아니며 작은 값에서는 `atol`을 직접 정해야 합니다.

##### 관련 수식과 코드 연결

$$
\lvert a_i-b_i\rvert \le \mathrm{atol}+\mathrm{rtol}\,\lvert b_i\rvert
$$

- **기호:** $a_i$는 계산값, $b_i$는 기준값, `atol`은 절대 오차, `rtol`은 상대 오차 허용치입니다.
- **수식 → 코드:** `isclose`는 이 부등식을 원소마다 계산하고, `allclose`는 모든 결과에 논리 AND를 적용합니다.
- **직관:** 완전히 같은 비트인지가 아니라 실험 목적에 충분히 가까운지를 묻는 검사입니다.
- **shape 확인:** 두 입력은 broadcasting 가능해야 합니다. `isclose`는 broadcast shape, `allclose`는 scalar `bool`을 반환합니다.
- **공식 문서:** [torch.allclose](https://docs.pytorch.org/docs/stable/generated/torch.allclose.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
class RelativePositionBias(nn.Module):
    """Clipped relative distance를 head별 attention bias로 바꾼다."""

    def __init__(self, heads, max_distance):
        super().__init__()
        self.max_distance = max_distance
        self.embedding = nn.Embedding(2 * max_distance + 1, heads)

    def forward(self, query_length, key_length, device):
        query = torch.arange(query_length, device=device)[:, None]
        key = torch.arange(key_length, device=device)[None, :]
        relative = (key - query).clamp(
            -self.max_distance,
            self.max_distance,
        )
        indices = relative + self.max_distance
        return self.embedding(indices).permute(2, 0, 1)


class ExplicitMultiHeadAttention(nn.Module):
    """QK^T/sqrt(d) 위에 bias와 mask를 직접 적용한다."""

    def __init__(self, model_dim, heads):
        super().__init__()
        if model_dim % heads != 0:
            raise ValueError("model_dim must be divisible by heads")
        self.heads = heads
        self.head_dim = model_dim // heads
        self.query_projection = nn.Linear(model_dim, model_dim)
        self.key_projection = nn.Linear(model_dim, model_dim)
        self.value_projection = nn.Linear(model_dim, model_dim)
        self.output_projection = nn.Linear(model_dim, model_dim)

    def split_heads(self, tensor):
        batch, length, _ = tensor.shape
        tensor = tensor.reshape(
            batch,
            length,
            self.heads,
            self.head_dim,
        )
        return tensor.transpose(1, 2)

    def merge_heads(self, tensor):
        batch, _, length, _ = tensor.shape
        tensor = tensor.transpose(1, 2).contiguous()
        return tensor.reshape(
            batch,
            length,
            self.heads * self.head_dim,
        )

    def forward(
        self,
        query,
        key,
        value,
        relative_bias=None,
        attention_mask=None,
    ):
        queries = self.split_heads(self.query_projection(query))
        keys = self.split_heads(self.key_projection(key))
        values = self.split_heads(self.value_projection(value))
        scale = self.head_dim**-0.5
        scores = torch.matmul(queries, keys.transpose(-2, -1)) * scale
        if relative_bias is not None:
            scores = scores + relative_bias[None]
        if attention_mask is not None:
            scores = scores.masked_fill(
                attention_mask[None, None],
                float("-inf"),
            )
        weights = scores.softmax(dim=-1)
        context = torch.matmul(weights, values)
        output = self.output_projection(self.merge_heads(context))
        return output, weights


class T5EncoderBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.norm1 = nn.LayerNorm(config.model_dim)
        self.attention = ExplicitMultiHeadAttention(
            config.model_dim,
            config.heads,
        )
        self.norm2 = nn.LayerNorm(config.model_dim)
        self.feedforward = nn.Sequential(
            nn.Linear(config.model_dim, config.feedforward_dim),
            nn.ReLU(),
            nn.Linear(config.feedforward_dim, config.model_dim),
        )

    def forward(self, hidden, relative_bias):
        normalized = self.norm1(hidden)
        attended, weights = self.attention(
            normalized,
            normalized,
            normalized,
            relative_bias=relative_bias,
        )
        hidden = hidden + attended
        hidden = hidden + self.feedforward(self.norm2(hidden))
        return hidden, weights


class T5DecoderBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.norm1 = nn.LayerNorm(config.model_dim)
        self.self_attention = ExplicitMultiHeadAttention(
            config.model_dim,
            config.heads,
        )
        self.norm2 = nn.LayerNorm(config.model_dim)
        self.cross_attention = ExplicitMultiHeadAttention(
            config.model_dim,
            config.heads,
        )
        self.norm3 = nn.LayerNorm(config.model_dim)
        self.feedforward = nn.Sequential(
            nn.Linear(config.model_dim, config.feedforward_dim),
            nn.ReLU(),
            nn.Linear(config.feedforward_dim, config.model_dim),
        )

    def forward(self, hidden, memory, relative_bias, causal_mask):
        normalized = self.norm1(hidden)
        attended, self_weights = self.self_attention(
            normalized,
            normalized,
            normalized,
            relative_bias=relative_bias,
            attention_mask=causal_mask,
        )
        hidden = hidden + attended
        normalized = self.norm2(hidden)
        attended, cross_weights = self.cross_attention(
            normalized,
            memory,
            memory,
        )
        hidden = hidden + attended
        hidden = hidden + self.feedforward(self.norm3(hidden))
        return hidden, self_weights, cross_weights


class T5Mini(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, config.model_dim)
        self.encoder_relative_bias = RelativePositionBias(
            config.heads,
            config.max_distance,
        )
        self.decoder_relative_bias = RelativePositionBias(
            config.heads,
            config.max_distance,
        )
        self.encoder = T5EncoderBlock(config)
        self.decoder = T5DecoderBlock(config)
        self.output = nn.Linear(config.model_dim, vocab_size)

    def causal_mask(self, length, device):
        return torch.ones(
            length,
            length,
            dtype=torch.bool,
            device=device,
        ).triu(diagonal=1)

    def forward(self, source, decoder_inputs):
        source_hidden = self.embedding(source)
        source_bias = self.encoder_relative_bias(
            source.shape[1],
            source.shape[1],
            source.device,
        )
        memory, _ = self.encoder(source_hidden, source_bias)
        target_hidden = self.embedding(decoder_inputs)
        target_length = decoder_inputs.shape[1]
        target_bias = self.decoder_relative_bias(
            target_length,
            target_length,
            source.device,
        )
        target_mask = self.causal_mask(target_length, source.device)
        decoded, _, _ = self.decoder(
            target_hidden,
            memory,
            target_bias,
            target_mask,
        )
        return self.output(decoded)


model = T5Mini(VOCAB_SIZE, config).to(DEVICE)
bias = model.encoder_relative_bias(2, 4, DEVICE)
assert bias.shape == (4, 2, 4)
train_logits = model(train_span_input, train_decoder_inputs)
assert train_logits.shape == (48, 4, VOCAB_SIZE)
probe = train_logits[0, -1, SENTINEL]
probe.backward()
encoder_gradient = model.encoder_relative_bias.embedding.weight.grad
decoder_gradient = model.decoder_relative_bias.embedding.weight.grad
assert encoder_gradient is not None
assert decoder_gradient is not None
assert float(encoder_gradient.abs().sum()) > 0.0
assert float(decoder_gradient.abs().sum()) > 0.0
model.zero_grad(set_to_none=True)

changed_inputs = train_decoder_inputs[:2].clone()
changed_inputs[:, -1] = EOS
with torch.no_grad():
    original_logits = model(
        train_span_input[:2],
        train_decoder_inputs[:2],
    )
    changed_logits = model(train_span_input[:2], changed_inputs)
assert torch.allclose(
    original_logits[:, :-1],
    changed_logits[:, :-1],
    atol=1e-6,
)

### 구현 단계 3 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §3.3.4: span-corruption objective
- **노트북에서 구현할 부분:** Task 3
- **구현 이유:** 원문의 확률 분해나 tensor 변환을 작은 입력에서 직접
  관찰하고, 고수준 API 뒤에 핵심 알고리즘을 숨기지 않기 위해서다.
- **완료 증거:** denoising accuracy

아래 TODO 전에 입력 token/feature shape와 수식의 축을 먼저 확인한다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=F.cross_entropy,Module.train,Optimizer.step,Tensor.detach,Module.eval,torch.optim.Adam,Module.parameters -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 7개 · 수식 3개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>F.cross_entropy(input, target, ...)</code></summary>

#### `F.cross_entropy(input, target, ...)`

- **역할:** logit에 log-softmax와 음의 로그우도를 결합합니다.
- **입력·반환:** 예측·target과 설정을 받고 축약된 loss 또는 같은 shape의 loss를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 objective를 수치화해 역전파 시작점으로 사용합니다.
- **주의점:** 입력 형식, target dtype·shape, `reduction` 의미를 반드시 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\mathcal{L}_{CE}=-\frac{1}{N}\sum_{n=1}^{N}\log\frac{e^{z_{n,y_n}}}{\sum_{c=1}^{C}e^{z_{n,c}}}
$$

- **기호:** $z_{n,c}$는 class logit, $y_n$은 정답 class ID, $C$는 class 수입니다.
- **수식 → 코드:** 함수가 내부에서 `log_softmax`와 정답 class의 음의 로그우도를 결합합니다.
- **직관:** 정답 class 확률이 낮을수록 큰 벌점을 줍니다.
- **shape 확인:** 일반적으로 logits `[N,C,...]`, class-index target `[N,...]`을 사용합니다.
- **공식 문서:** [F.cross_entropy](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html)

</details>

<details>
<summary><code>model.train(mode=True)</code></summary>

#### `model.train(mode=True)`

- **역할:** 모듈과 하위 모듈을 학습 모드로 전환합니다.
- **입력·반환:** 불리언 모드를 받고 모듈 자신을 반환합니다.
- **이 셀에서 쓰는 이유:** Dropout과 BatchNorm이 학습 시 동작을 사용하도록 합니다.
- **주의점:** gradient를 켜는 함수가 아니므로 `no_grad`와 독립적입니다.
- **공식 문서:** [Module.train](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>optimizer.step(closure=None)</code></summary>

#### `optimizer.step(closure=None)`

- **역할:** 현재 gradient와 optimizer 규칙으로 파라미터를 갱신합니다.
- **입력·반환:** 선택적 closure를 받고 optimizer에 따라 loss 또는 `None`을 반환합니다.
- **이 셀에서 쓰는 이유:** 역전파로 계산한 gradient를 실제 학습 update에 반영합니다.
- **주의점:** 일반적으로 `zero_grad`, forward, `backward` 다음에 호출합니다.

##### 관련 수식과 코드 연결

$$
\theta_{t+1}=\mathcal{U}(\theta_t,\nabla_\theta L_t,s_t;\eta)
$$

- **기호:** $\mathcal U$는 optimizer update, $s_t$는 momentum·moment 상태입니다.
- **수식 → 코드:** `step()`은 이미 `.grad`에 저장된 값을 읽어 선택한 optimizer 규칙을 적용합니다.
- **직관:** gradient 계산과 parameter 갱신을 분리하면 학습 lifecycle을 정확히 추적할 수 있습니다.
- **shape 확인:** parameter shape은 유지되고 값과 optimizer state만 바뀝니다.
- **공식 문서:** [Optimizer.step](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.step.html)

</details>

<details>
<summary><code>tensor.detach()</code></summary>

#### `tensor.detach()`

- **역할:** 같은 저장공간을 공유하면서 현재 계산 그래프에서 분리한 텐서를 만듭니다.
- **입력·반환:** 텐서를 받아 gradient를 추적하지 않는 텐서 view를 반환합니다.
- **이 셀에서 쓰는 이유:** 로그·teacher target·NumPy 변환이 역전파 경로를 만들지 않게 합니다.
- **주의점:** 원본과 저장공간을 공유하므로 in-place 변경은 양쪽에 영향을 줄 수 있습니다.
- **공식 문서:** [Tensor.detach](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.detach.html)

</details>

<details>
<summary><code>model.eval()</code></summary>

#### `model.eval()`

- **역할:** 모듈과 하위 모듈을 평가 모드로 전환합니다.
- **입력·반환:** 인자 없이 호출하고 모듈 자신을 반환합니다.
- **이 셀에서 쓰는 이유:** Dropout을 끄고 BatchNorm의 저장 통계를 사용하게 합니다.
- **주의점:** gradient 기록은 끄지 않으므로 필요하면 `no_grad`도 사용합니다.
- **공식 문서:** [Module.eval](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>torch.optim.Adam(params, lr=1e-3, ...)</code></summary>

#### `torch.optim.Adam(params, lr=1e-3, ...)`

- **역할:** 1·2차 gradient 모멘트를 추적하는 Adam optimizer를 만듭니다.
- **입력·반환:** 파라미터와 학습률 등을 받고 optimizer 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 노트북의 trainable parameter를 반복 update합니다.
- **주의점:** weight decay의 해석은 AdamW와 다르며 학습률이 결과에 크게 작용합니다.

##### 관련 수식과 코드 연결

$$
\begin{aligned}m_t&=\beta_1m_{t-1}+(1-\beta_1)g_t,\\v_t&=\beta_2v_{t-1}+(1-\beta_2)g_t^2,\\\theta_t&=\theta_{t-1}-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}\end{aligned}
$$

- **기호:** $g_t$는 gradient, $m_t,v_t$는 1·2차 모멘트, $\eta$는 학습률입니다.
- **수식 → 코드:** `backward()`가 만든 gradient를 읽어 모멘트를 갱신한 뒤 `step()`에서 파라미터에 적용합니다.
- **직관:** 최근 gradient 방향과 크기를 함께 누적해 파라미터별 적응형 보폭을 만듭니다.
- **shape 확인:** 각 trainable parameter와 같은 shape의 모멘트 상태가 optimizer에 저장됩니다.
- **공식 문서:** [torch.optim.Adam](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html)

</details>

<details>
<summary><code>model.parameters(recurse=True)</code></summary>

#### `model.parameters(recurse=True)`

- **역할:** 모듈에 등록된 학습 파라미터 iterator를 반환합니다.
- **입력·반환:** 하위 모듈 포함 여부를 받고 `Parameter` iterator를 반환합니다.
- **이 셀에서 쓰는 이유:** optimizer가 갱신할 weight와 bias를 전달합니다.
- **주의점:** iterator는 한 번 소비되며 buffer와 일반 tensor attribute는 포함하지 않습니다.
- **공식 문서:** [Module.parameters](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
def denoising_loss(logits, targets):
    return F.cross_entropy(
        logits.reshape(-1, logits.shape[-1]),
        targets.reshape(-1),
    )


def train_t5_step(model, optimizer, source, decoder_inputs, targets):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    loss = denoising_loss(model(source, decoder_inputs), targets)
    loss.backward()
    optimizer.step()
    return float(loss.detach())


def evaluate_t5(model, source, decoder_inputs, targets):
    model.eval()
    with torch.no_grad():
        predictions = model(source, decoder_inputs).argmax(dim=-1)
    return float((predictions == targets).float().mean())


optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
losses = []
for step in range(config.steps):
    losses.append(
        train_t5_step(
            model,
            optimizer,
            train_span_input,
            train_decoder_inputs,
            train_span_target,
        )
    )
heldout_accuracy = evaluate_t5(
    model,
    test_span_input,
    test_decoder_inputs,
    test_span_target,
)
assert min(losses[3:]) < losses[0]
assert heldout_accuracy > 0.75
plt.figure(figsize=(5, 3))
plt.plot(losses)
plt.title(f"T5 held-out span accuracy={heldout_accuracy:.1%}")
plt.tight_layout()
plt.show()

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> C4와 최대 11B parameter 대신 60개 3-token sequence와 작은 Transformer를 쓰며, 48개 train/12개 held-out 행을 분리한다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.